<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# LogiTrack — Supply Chain Analytics
## Notebook 1 — Contexte métier & Exploration des données
### ✅ VERSION CORRIGÉE

> **Comment lire ce corrigé :**  
> Les blocs `MÉTHODE` expliquent les choix techniques et les patterns généralisables.  
> Les blocs `INTERPRÉTATION` lisent les résultats.  
> Les blocs `MÉTIER` font le lien entre le chiffre et la décision business.

| | |
|---|---|
| **Entreprise** | LogiTrack — opérateur logistique régional, Afrique de l'Ouest |
| **Période** | Janvier 2022 — Décembre 2023 |
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB (JupySQL), matplotlib |
| **Durée estimée** | 3h à 4h |

---
> LogiTrack achemine **50 000 colis par mois** entre 6 pays d'Afrique de l'Ouest avec 15 transporteurs partenaires et 12 entrepôts de transit. Le Directeur des Opérations fait face à un problème structurel :
>
> *"On gère les retards à la réaction — le client appelle, on cherche le colis, et on découvre qu'il est bloqué en douane depuis trois jours. À ce stade la pénalité est déjà déclenchée et le client a déjà perdu confiance."*

---
## 0. Mise en place de l'environnement

In [1]:
!pip install jupysql==0.11.1 duckdb-engine --quiet


[notice] A new release of pip is available: 24.1 -> 26.0.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import duckdb
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/logitrack_analytics/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

KeyboardInterrupt: 

---
## 1. Connexion DuckDB et chargement des 7 tables

### MÉTHODE
`read_csv_auto()` charge les CSV directement depuis GitHub, infère les types et crée les tables en une seule instruction — zéro configuration, fonctionne en Colab et en local.

> **Ordre de chargement :** les tables de référence d'abord (`clients`, `transporteurs`, `entrepots`, `routes`), puis les tables de faits (`livraisons`, `evenements`, `alertes_sla`). Cette convention facilite la lecture même si DuckDB n'enforce pas les FK.

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack_analytics/data/'

conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE clients       AS SELECT * FROM read_csv_auto('{BASE_URL}clients.csv');
    CREATE TABLE transporteurs AS SELECT * FROM read_csv_auto('{BASE_URL}transporteurs.csv');
    CREATE TABLE entrepots     AS SELECT * FROM read_csv_auto('{BASE_URL}entrepots.csv');
    CREATE TABLE routes        AS SELECT * FROM read_csv_auto('{BASE_URL}routes.csv');
    CREATE TABLE livraisons    AS SELECT * FROM read_csv_auto('{BASE_URL}livraisons.csv',    timestampformat='%Y-%m-%d');
    CREATE TABLE evenements    AS SELECT * FROM read_csv_auto('{BASE_URL}evenements.csv',    timestampformat='%Y-%m-%d %H:%M');
    CREATE TABLE alertes_sla   AS SELECT * FROM read_csv_auto('{BASE_URL}alertes_sla.csv',  timestampformat='%Y-%m-%d %H:%M');
""")

result = conn.execute("""
    SELECT 'clients'       AS table_name, COUNT(*) AS nb_lignes,    30 AS attendu FROM clients       UNION ALL
    SELECT 'transporteurs',               COUNT(*),                 15            FROM transporteurs UNION ALL
    SELECT 'entrepots',                   COUNT(*),                 12            FROM entrepots     UNION ALL
    SELECT 'routes',                      COUNT(*),                 30            FROM routes        UNION ALL
    SELECT 'livraisons',                  COUNT(*),              15000            FROM livraisons    UNION ALL
    SELECT 'evenements',                  COUNT(*),              28286            FROM evenements    UNION ALL
    SELECT 'alertes_sla',                 COUNT(*),               4972            FROM alertes_sla
""").df()
display(result)
print('✅ 7 tables chargées')

In [ ]:
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

---
## 2. Exploration des tables

### MÉTHODE
Avant d'analyser, il faut maîtriser les clés de jointure. Le schéma LogiTrack est en étoile autour de `livraisons` — toutes les dimensions pointent vers cette table centrale.

```
clients ──────────┐
transporteurs ────┤
entrepots (x2) ───┼──► livraisons ──► evenements
routes ───────────┘         │
                            └──► alertes_sla
```

> **Double jointure entrepôts :** `livraisons` référence `entrepots` via deux clés distinctes — `entrepot_origine_id` et `entrepot_dest_id`. Utiliser des alias (`e_orig` et `e_dest`) pour éviter l'ambiguïté.

In [ ]:
%%sql
SELECT * FROM livraisons LIMIT 5

In [ ]:
%%sql
SELECT * FROM transporteurs

In [ ]:
%%sql
SELECT * FROM routes ORDER BY risque_douanier DESC, distance_km DESC LIMIT 10

In [ ]:
%%sql
-- Schéma complet : toutes les colonnes de toutes les tables
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'main'
ORDER BY table_name, ordinal_position

> **INTERPRÉTATION — Structure des 7 tables :**
>
> - **livraisons** : 32 colonnes — table centrale avec `livraison_en_retard` (variable cible ML), `sla_breach`, `escalade`, `in_backlog`, `penalite_fcfa`
> - **transporteurs** : 9 colonnes — `taux_livraison_temps` et `tarif_moyen_fcfa_100kg` permettront la matrice coût × fiabilité
> - **routes** : 8 colonnes — `risque_douanier` (Faible/Moyen/Élevé) est un prédicteur fort du retard
> - **evenements** : couverture partielle — 28 286 événements sur 8 000 livraisons (LEFT JOIN obligatoire)
> - **alertes_sla** : 4 972 alertes sur les 5 853 breaches attendus (85% de couverture)

---
## 3. Diagnostic qualité

### MÉTHODE
Le diagnostic qualité suit toujours le même ordre : nulls → doublons → aberrants → incohérences métier. Le pattern `SUM(CASE WHEN condition THEN 1 ELSE 0 END)` compte les anomalies sans filtrer les données — on garde la vision globale.

In [ ]:
%%sql df_diag_liv <<
-- DIAGNOSTIC 1 : Anomalies dans la table livraisons
SELECT
    COUNT(*)                                                          AS total_lignes,
    SUM(CASE WHEN client_id       IS NULL THEN 1 ELSE 0 END)          AS null_client,
    SUM(CASE WHEN transporteur_id IS NULL THEN 1 ELSE 0 END)          AS null_transporteur,
    SUM(CASE WHEN poids_kg        <= 0    THEN 1 ELSE 0 END)          AS poids_aberrants,
    SUM(CASE WHEN csat IS NOT NULL
             AND (csat < 1 OR csat > 5)  THEN 1 ELSE 0 END)          AS csat_hors_plage,
    SUM(CASE WHEN delai_jours     < -30   THEN 1 ELSE 0 END)          AS delais_aberrants,
    SUM(CASE WHEN date_livraison_reelle < date_enlevement
             AND date_livraison_reelle IS NOT NULL
                                         THEN 1 ELSE 0 END)          AS incoherences_dates,
    SUM(CASE WHEN statut = 'Livré' AND sla_breach = 1 THEN 1 ELSE 0 END) AS livres_en_breach
FROM livraisons

> **INTERPRÉTATION — Anomalies livraisons :**
>
> | Anomalie | Nb | Traitement NB2 |
> |---|---|---|
> | Poids aberrants (≤ 0) | **2** | Imputation par médiane du type_colis |
> | CSAT hors plage [1-5] | **3** | Remplacement par NULL |
> | Délais aberrants (-99j) | **4** | Correction par `duree_reelle_jours - sla_jours_contractuel` |
>
> **MÉTIER :** Ces 9 anomalies représentent 0.06% des livraisons — impact faible sur les KPIs globaux mais critique pour le ML où chaque valeur aberrante peut biaiser l'entraînement.

In [ ]:
%%sql df_diag_trp <<
-- DIAGNOSTIC 2 : Transporteurs inactifs ou suspendus utilisés
SELECT
    t.transporteur_id,
    t.nom,
    t.statut,
    COUNT(l.livraison_id)                                             AS nb_livraisons,
    ROUND(SUM(l.sla_breach) * 100.0 / COUNT(*), 1)                   AS taux_breach_pct
FROM transporteurs t
LEFT JOIN livraisons l ON t.transporteur_id = l.transporteur_id
WHERE t.statut != 'Actif' OR t.actif = false
GROUP BY t.transporteur_id, t.nom, t.statut

> **INTERPRÉTATION — Transporteur suspendu :**
>
> **TRP15 (QuickDeliver)** est référencé comme `Suspendu` dans le référentiel mais apparaît sur des livraisons réelles. Ces livraisons seront **exclues** des analyses de performance en NB2 (`WHERE t.actif = true`) — un partenaire suspendu ne doit pas biaiser le benchmarking des transporteurs actifs.
>
> **MÉTIER :** La présence de TRP15 indique un défaut de synchronisation entre le système de gestion des partenaires et la plateforme opérationnelle. Une alerte automatique devrait bloquer toute nouvelle mission assignée à un transporteur suspendu.

In [ ]:
%%sql df_diag_dup <<
-- DIAGNOSTIC 3 : Doublons sur livraison_id
SELECT
    COUNT(*)                                                          AS total_livraisons,
    COUNT(DISTINCT livraison_id)                                      AS livraisons_uniques,
    COUNT(*) - COUNT(DISTINCT livraison_id)                           AS doublons
FROM livraisons

In [ ]:
%%sql df_diag_statuts <<
-- DIAGNOSTIC 4 : Cohérence statuts vs dates
SELECT
    statut,
    COUNT(*)                                                          AS nb,
    SUM(CASE WHEN date_livraison_reelle IS NULL THEN 1 ELSE 0 END)    AS sans_date_reelle,
    SUM(CASE WHEN date_livraison_reelle IS NOT NULL THEN 1 ELSE 0 END) AS avec_date_reelle,
    ROUND(SUM(sla_breach) * 100.0 / COUNT(*), 1)                     AS pct_breach
FROM livraisons
GROUP BY statut
ORDER BY nb DESC

> **INTERPRÉTATION — Cohérence statuts :**
>
> - `Livré` et `Livré avec retard` ont tous une `date_livraison_reelle` renseignée ✅
> - `Perdu` et `En transit` ont une `date_livraison_reelle` NULL ✅ — cohérent
> - `Livré avec retard` : 100% de `sla_breach = 1` ✅
> - `Livré` : quelques `sla_breach = 1` possibles — cas où le colis est arrivé à temps mais le SLA a été recalculé
>
> **MÉTIER :** 117 colis perdus (0.78%) représentent le cas le plus grave — perte totale du colis, remboursement client intégral et impact réputationnel fort.

In [ ]:
%%sql df_diag_recap <<
-- DIAGNOSTIC 5 : Résumé synthétique toutes tables
SELECT 'livraisons'    AS table_name, 15000 AS attendu, COUNT(*) AS reel FROM livraisons    UNION ALL
SELECT 'evenements',                  28286,             COUNT(*) FROM evenements    UNION ALL
SELECT 'alertes_sla',                  4972,             COUNT(*) FROM alertes_sla   UNION ALL
SELECT 'clients',                        30,             COUNT(*) FROM clients        UNION ALL
SELECT 'transporteurs',                  15,             COUNT(*) FROM transporteurs  UNION ALL
SELECT 'entrepots',                      12,             COUNT(*) FROM entrepots      UNION ALL
SELECT 'routes',                         30,             COUNT(*) FROM routes

---
## 4. Premiers KPIs opérationnels

### KPI 1 — Vue globale

### MÉTHODE
Une seule requête pour tous les KPIs globaux — garantit la cohérence des chiffres. `ROUND(SUM(sla_breach) * 100.0 / COUNT(*), 1)` est le pattern standard pour un taux en pourcentage. `NULLIF(COUNT(*), 0)` protège contre la division par zéro sur des sous-ensembles filtrés.

In [ ]:
%%sql df_kpi_global <<
SELECT
    COUNT(*)                                                          AS total_livraisons,
    SUM(sla_breach)                                                   AS nb_breach,
    ROUND(SUM(sla_breach) * 100.0 / COUNT(*), 1)                     AS taux_breach_pct,
    ROUND(AVG(CASE WHEN csat IS NOT NULL
                    AND csat BETWEEN 1 AND 5 THEN csat END), 2)       AS csat_moyen,
    SUM(escalade)                                                     AS nb_escalades,
    ROUND(SUM(escalade) * 100.0 / COUNT(*), 1)                       AS taux_escalade_pct,
    SUM(penalite_fcfa)                                                AS total_penalites_fcfa,
    ROUND(AVG(CASE WHEN sla_breach = 1 THEN delai_jours END), 1)     AS delai_moyen_retard_j,
    SUM(CASE WHEN statut = 'Perdu' THEN 1 ELSE 0 END)                AS nb_colis_perdus,
    ROUND(SUM(CASE WHEN statut = 'Perdu' THEN 1 ELSE 0 END)
          * 100.0 / COUNT(*), 2)                                     AS taux_perte_pct
FROM livraisons
WHERE poids_kg > 0
  AND (csat IS NULL OR csat BETWEEN 1 AND 5)

> **INTERPRÉTATION — KPIs globaux :**
>
> | KPI | Valeur | Lecture |
> |---|---|---|
> | Total livraisons | **15 000** | Volume 2022-2023 |
> | Taux SLA breach | **39.0%** | 4 livraisons sur 10 dépassent le délai |
> | CSAT moyen | **3.65 / 5** | Satisfaction dégradée par les retards |
> | Taux escalade | **6.1%** | 915 tickets remontés au management |
> | Délai moyen retard | **6.3 jours** | Retard structurel, pas marginal |
> | Colis perdus | **117 (0.78%)** | Perte totale — remboursement intégral |
>
> **MÉTIER :** Un taux de breach à 39% est alarmant — presque 2 livraisons sur 5 ne respectent pas le délai contractuel. Pour les clients Grands Comptes avec pénalités, chaque jour de retard supplémentaire déclenche des frais. La priorité est d'identifier les corridors et transporteurs qui concentrent ce problème.

---
### KPI 2 — Taux de breach par corridor

### MÉTHODE
`RANK() OVER (ORDER BY taux_breach DESC)` classe les corridors sans réduire le nombre de lignes. `CONCAT(pays_origine, ' → ', pays_destination)` crée un libellé lisible directement exploitable dans Power BI.

In [ ]:
%%sql df_kpi_corridors <<
SELECT
    l.pays_origine || ' → ' || l.pays_destination                    AS corridor,
    r.risque_douanier,
    r.distance_km,
    COUNT(l.livraison_id)                                             AS nb_livraisons,
    ROUND(SUM(l.sla_breach) * 100.0 / COUNT(*), 1)                   AS taux_breach_pct,
    ROUND(AVG(CASE WHEN l.sla_breach = 1
              THEN l.delai_jours END), 1)                            AS delai_moy_retard_j,
    SUM(l.penalite_fcfa)                                              AS penalites_fcfa,
    RANK() OVER (ORDER BY SUM(l.sla_breach) * 100.0
                          / COUNT(*) DESC)                           AS rang_risque
FROM livraisons l
JOIN routes r ON l.route_id = r.route_id
GROUP BY l.pays_origine, l.pays_destination, r.risque_douanier, r.distance_km
ORDER BY rang_risque
LIMIT 10

> **INTERPRÉTATION — Top corridors en breach :**
>
> Les corridors impliquant le **Mali** et le **Burkina Faso** concentrent les taux de breach les plus élevés — cohérent avec leur risque douanier `Élevé` et la performance faible de TRP05, TRP06 et TRP12 qui opèrent principalement sur ces routes.
>
> **MÉTIER :** Un corridor à risque `Élevé` ne signifie pas qu'il faut l'abandonner — il signifie qu'il faut des transporteurs plus fiables ou des SLA recalibrés pour refléter la réalité opérationnelle. Proposer un SLA de 5 jours sur un corridor qui prend structurellement 8 jours est une promesse impossible à tenir.

---
### KPI 3 — Performance des transporteurs

### MÉTHODE
On exclut TRP15 (suspendu) de l'analyse de performance. Le ratio `cout_total / nb_livraisons` donne le coût moyen par livraison — plus parlant que le tarif au 100kg pour comparer des transporteurs aux volumes différents.

In [ ]:
%%sql df_kpi_trp <<
SELECT
    t.transporteur_id,
    t.nom,
    t.mode_transport,
    COUNT(l.livraison_id)                                             AS nb_livraisons,
    ROUND(SUM(l.sla_breach) * 100.0 / COUNT(*), 1)                   AS taux_breach_pct,
    ROUND(AVG(CASE WHEN l.csat BETWEEN 1 AND 5
              THEN l.csat END), 2)                                   AS csat_moyen,
    ROUND(AVG(l.duree_reelle_jours), 1)                              AS duree_moy_j,
    SUM(l.penalite_fcfa)                                              AS penalites_fcfa,
    ROUND(AVG(l.cout_transport_fcfa), 0)                             AS cout_moy_fcfa,
    RANK() OVER (ORDER BY SUM(l.sla_breach) * 100.0
                          / COUNT(*) ASC)                            AS rang_fiabilite
FROM transporteurs t
JOIN livraisons l ON t.transporteur_id = l.transporteur_id
WHERE t.actif = true
GROUP BY t.transporteur_id, t.nom, t.mode_transport
ORDER BY rang_fiabilite

> **INTERPRÉTATION — Classement des transporteurs :**
>
> - **TRP08 (AirFreight Premium)** : taux breach proche de 0, CSAT excellent — mais coût 4× supérieur à la moyenne. Réservé aux envois urgents.
> - **TRP07 (TransOuest)** et **TRP03 (CamLog)** : meilleur rapport fiabilité/coût du réseau — à privilégier sur les corridors Côte d'Ivoire-Cameroun.
> - **TRP12 (NordSud Express)** : taux breach le plus élevé du réseau, coût parmi les plus bas. Moins cher à court terme, mais les pénalités clients effacent l'économie.
>
> **MÉTIER :** Le vrai indicateur de performance d'un transporteur n'est pas son tarif mais son **coût total incluant les pénalités**. TRP12 est le moins cher à l'expédition mais le plus coûteux en valeur réelle.

---
### KPI 4 — Volume mensuel et saisonnalité

### MÉTHODE
`strftime(date_creation, '%Y-%m')` extrait l'année-mois depuis une date — équivalent de `dt.to_period('M')` en pandas. `LAG()` sur la série mensuelle permet de calculer la variation mois par mois dès ce notebook.

In [ ]:
%%sql df_kpi_mensuel <<
WITH mensuel AS (
    SELECT
        strftime(date_creation, '%Y-%m')                              AS mois,
        COUNT(*)                                                      AS nb_livraisons,
        SUM(sla_breach)                                               AS nb_breach,
        ROUND(SUM(sla_breach) * 100.0 / COUNT(*), 1)                 AS taux_breach_pct,
        ROUND(AVG(CASE WHEN csat BETWEEN 1 AND 5 THEN csat END), 2)  AS csat_moy
    FROM livraisons
    GROUP BY strftime(date_creation, '%Y-%m')
)
SELECT
    mois,
    nb_livraisons,
    nb_breach,
    taux_breach_pct,
    csat_moy,
    LAG(nb_livraisons) OVER (ORDER BY mois)                          AS vol_mois_prec,
    ROUND((nb_livraisons - LAG(nb_livraisons) OVER (ORDER BY mois))
          * 100.0
          / NULLIF(LAG(nb_livraisons) OVER (ORDER BY mois), 0), 1)   AS variation_vol_pct
FROM mensuel
ORDER BY mois

> **INTERPRÉTATION — Saisonnalité :**
>
> - **Novembre-Janvier** : pic de volume (+30 à +50% vs moyenne) — période des fêtes et fins d'année fiscales
> - **Juillet-Août** : creux estival (-20% vs moyenne)
> - Le taux de breach suit la même saisonnalité — plus de volume = moins de capacité disponible = plus de retards
>
> **MÉTIER :** Le pic de novembre-janvier est prévisible mais LogiTrack n'a pas de capacité buffer pour l'absorber. Pré-réserver des créneaux transporteurs en octobre permettrait de lisser la charge et de réduire le taux de breach sur cette période critique.

---
### KPI 5 — Performance des entrepôts

### MÉTHODE
On joint `livraisons` à `entrepots` deux fois avec des alias distincts — `e_orig` pour l'entrepôt de départ et `e_dest` pour l'entrepôt d'arrivée. Analyser les deux perspectives révèle où le retard est créé dans la chaîne.

In [ ]:
%%sql df_kpi_ent <<
SELECT
    e_orig.nom                                                        AS entrepot_origine,
    e_orig.type                                                       AS type_entrepot,
    e_orig.taux_ponctualite                                           AS ponctualite_declaree,
    COUNT(l.livraison_id)                                             AS nb_livraisons_depart,
    ROUND(SUM(l.sla_breach) * 100.0 / COUNT(*), 1)                   AS taux_breach_depuis,
    ROUND(AVG(l.duree_reelle_jours), 1)                              AS duree_moy_j
FROM livraisons l
JOIN entrepots e_orig ON l.entrepot_origine_id = e_orig.entrepot_id
JOIN entrepots e_dest ON l.entrepot_dest_id    = e_dest.entrepot_id
GROUP BY e_orig.entrepot_id, e_orig.nom, e_orig.type, e_orig.taux_ponctualite
ORDER BY taux_breach_depuis DESC

> **INTERPRÉTATION — Entrepôts :**
>
> Les entrepôts de **Bamako** (ENT09) et **Ouagadougou** (ENT10) affichent les taux de breach au départ les plus élevés, cohérents avec leur `taux_ponctualite` déclaré (0.75-0.77). Ces hubs sahéliens sont des goulots d'étranglement structurels.
>
> **MÉTIER :** Un entrepôt avec un `taux_ponctualite` de 0.75 signifie qu'1 départ sur 4 est déjà en retard avant même que le transporteur ne prenne le colis. Renforcer les équipes de dispatch sur ces sites aurait un impact direct sur le taux de breach global.

---
## 5. Visualisation 2×2

### MÉTHODE
Une figure 2×2 présente 4 dimensions complémentaires en une seule image. On utilise `COLORS` pour l'identité visuelle DataProjectLab et on sauvegarde dans `SAVE_PATH` pour réutilisation dans Power BI.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LogiTrack — Vue d\'ensemble opérationnelle', fontsize=16,
             fontweight='bold', color=COLORS['primary'], y=1.01)

# 1. Volume mensuel + taux breach
ax1 = axes[0, 0]
ax1b = ax1.twinx()
ax1.bar(df_kpi_mensuel['mois'], df_kpi_mensuel['nb_livraisons'],
        color=COLORS['primary'], alpha=0.6, label='Volume')
ax1b.plot(df_kpi_mensuel['mois'], df_kpi_mensuel['taux_breach_pct'],
          color=COLORS['danger'], linewidth=2, marker='o', markersize=4, label='Breach %')
ax1.set_title('Volume mensuel & Taux breach', fontweight='bold')
ax1.set_ylabel('Nb livraisons')
ax1b.set_ylabel('Taux breach (%)')
ax1.tick_params(axis='x', rotation=45)
ax1.set_xticks(range(0, len(df_kpi_mensuel), 3))
ax1.set_xticklabels(df_kpi_mensuel['mois'].iloc[::3], rotation=45)

# 2. Top 10 transporteurs taux breach
ax2 = axes[0, 1]
trp_sorted = df_kpi_trp.sort_values('taux_breach_pct', ascending=True).tail(10)
colors_bars = [COLORS['danger'] if t > 45 else COLORS['warning'] if t > 35
               else COLORS['secondary'] for t in trp_sorted['taux_breach_pct']]
ax2.barh(trp_sorted['nom'], trp_sorted['taux_breach_pct'], color=colors_bars)
ax2.axvline(df_kpi_global['taux_breach_pct'].iloc[0], color=COLORS['neutral'],
            linestyle='--', linewidth=1.5, label='Moyenne')
ax2.set_title('Taux breach par transporteur', fontweight='bold')
ax2.set_xlabel('Taux breach (%)')
ax2.legend(fontsize=9)

# 3. Top 10 corridors taux breach
ax3 = axes[1, 0]
corr_sorted = df_kpi_corridors.sort_values('taux_breach_pct', ascending=True)
colors_corr = [COLORS['danger'] if r == 'Élevé' else COLORS['warning'] if r == 'Moyen'
               else COLORS['secondary'] for r in corr_sorted['risque_douanier']]
ax3.barh(corr_sorted['corridor'], corr_sorted['taux_breach_pct'], color=colors_corr)
ax3.set_title('Taux breach par corridor (top 10)', fontweight='bold')
ax3.set_xlabel('Taux breach (%)')

# 4. Distribution CSAT
ax4 = axes[1, 1]
df_csat = conn.execute("""
    SELECT ROUND(csat) AS note, COUNT(*) AS nb
    FROM livraisons
    WHERE csat BETWEEN 1 AND 5
    GROUP BY ROUND(csat)
    ORDER BY note
""").df()
ax4.bar(df_csat['note'].astype(int), df_csat['nb'],
        color=COLORS['primary'], alpha=0.85)
ax4.set_title('Distribution CSAT', fontweight='bold')
ax4.set_xlabel('Note (1-5)')
ax4.set_ylabel('Nb livraisons')
ax4.set_xticks([1, 2, 3, 4, 5])

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}logitrack_vue_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}logitrack_vue_ensemble.png')

> **INTERPRÉTATION — Vue d'ensemble :**
>
> **1. Volume & breach :** la saisonnalité est nette — le taux de breach monte en même temps que le volume (nov-jan). Le réseau n'absorbe pas le pic sans dégradation de qualité.
>
> **2. Transporteurs :** TRP12, TRP05 et TRP06 (corridors sahéliens) ont des taux de breach nettement au-dessus de la moyenne. TRP08 (aérien) est quasi-parfait mais non scalable.
>
> **3. Corridors :** les routes à risque douanier `Élevé` (rouge) dominent le top des breaches — la corrélation risque douanier × retard est forte et exploitable en ML.
>
> **4. CSAT :** distribution bimodale — forte concentration sur 5 (clients satisfaits des livraisons rapides) et sur 1-2 (clients pénalisés par les retards). Peu de 3, les clients polarisent leur jugement.

---
## Bilan du Notebook 1

### Anomalies identifiées

| Anomalie | Nb | Table |
|---|---|---|
| Poids kg aberrants (≤ 0) | **2** | livraisons |
| CSAT hors plage [1-5] | **3** | livraisons |
| Délais aberrants (-99j) | **4** | livraisons |
| Transporteur suspendu avec livraisons | **1** | transporteurs |

### KPIs de référence

| KPI | Valeur |
|---|---|
| Total livraisons | **15 000** |
| Taux SLA breach global | **39.0%** |
| CSAT moyen | **3.65 / 5** |
| Délai moyen retard | **6.3 jours** |
| Taux escalade | **6.1%** |
| Colis perdus | **117 (0.78%)** |
| Top 3 transporteurs breach | **TRP12, TRP05, TRP06** |
| `livraison_en_retard` (cible ML) | **39.2%** |

**Pour le Notebook 2 :** corriger les 9 anomalies, créer les features ML (`delai_relatif_sla`, `taux_breach_corridor`, `taux_breach_transporteur`, variables temporelles) et exporter `logitrack_analytics.csv`.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.